In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

TARGET_COL = "purchase_bid"  # change to "sell_bid" for sell-side experiments
LAG_SUFFIX = "pb" if TARGET_COL == "purchase_bid" else "sb"
HORIZON = 96 * 7
PARAMS_PATH = Path("data/processed/params/lstm_params.json")

from forecasting.data.loaders import load_market_data
from forecasting.features.calendar import build_time_features
from forecasting.models.lstm_models import build_default_lstm
from forecasting.utils.io import save_json, load_json

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from utilsforecast.plotting import plot_series
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import *

from sklearn.preprocessing import MinMaxScaler

from skforecast.deep_learning import ForecasterRnn
from keras.layers import Dropout
from keras.callbacks import EarlyStopping

from skforecast.model_selection import TimeSeriesFold, bayesian_search_forecaster, backtesting_forecaster, backtesting_forecaster_multiseries

In [ ]:
import os
os.environ["KERAS_BACKEND"]

In [ ]:
data_indexed = load_market_data("data/raw/iex-dam-0201-0421.csv")
data = data_indexed.reset_index()
data_indexed

In [ ]:
data_clean = data.copy()
data_clean["period_enum"] = (
    data_clean["period_start"].dt.hour * 4 + data_clean["period_start"].dt.minute // 15 + 1
)
data_clean["weekday_enum"] = data_clean["period_start"].dt.weekday + 1
data_clean[f"{LAG_SUFFIX}_lag1d"] = data_clean[TARGET_COL].shift(96 * 1)
data_clean[f"{LAG_SUFFIX}_lag2d"] = data_clean[TARGET_COL].shift(96 * 2)
data_clean[f"{LAG_SUFFIX}_lag7d"] = data_clean[TARGET_COL].shift(96 * 7)
data_clean = data_clean.dropna().reset_index(drop=True)

In [ ]:
# Split train-validation-test
# ==============================================================================
end_train = "2026-03-28 23:46:00"
end_validation = "2026-04-05 23:46:00"
data_train = data_indexed.loc[:end_train, :].copy()
data_val = data_indexed.loc[end_train:end_validation, :].copy()
data_test = data_indexed.loc[end_validation:, :].copy()

print(
    f"Dates train     : {data_train.index.min()} --- " 
    f"{data_train.index.max()}  (n={len(data_train)})"
)
print(
    f"Dates validation: {data_val.index.min()} --- " 
    f"{data_val.index.max()}  (n={len(data_val)})"
)
print(
    f"Dates test: {data_test.index.min()} --- " 
    f"{data_test.index.max()}  (n={len(data_test)})"
)

In [ ]:
exogenous_features = build_time_features(data_indexed.index)
exogenous_features.head(3)

In [ ]:
# Create model
# ==============================================================================
lags = 96

model = build_default_lstm(series=data[[TARGET_COL]], lags=lags, steps=HORIZON)

model.summary()

In [ ]:

forecaster = ForecasterRnn(
    estimator          = model,
    levels             = TARGET_COL,
    lags               = 96,
    transformer_series = MinMaxScaler(),
    fit_kwargs = {
        "epochs": 50,
        "batch_size": 32, # Smaller batch size for smaller datasets
        "callbacks": [EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)],
        "series_val": data_val # Your 7-day validation window
    }
)

# 3. Fit using your Cyclical Exogenous features
forecaster.fit(
    series = data_train[[TARGET_COL]],
    #exog   = exogenous_features[['hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos']]
)

In [ ]:
# Prediction
# ==============================================================================
predictions = forecaster.predict(steps=96)  # Same as steps=None
predictions.head(4)

In [ ]:
forecaster.max_step

In [ ]:
# Backtesting 
# ==============================================================================
cv = TimeSeriesFold(
         steps              = forecaster.max_step,
         initial_train_size = len(data_indexed.loc[:end_validation, :]),  # Training + Validation Data
         refit              = False
     )

# The validation partition is now used as part of the initial fit
# Epocs is set to the value idenfied in the previous training with early stopping
forecaster.set_fit_kwargs({"epochs": 10,"batch_size": 512})

metrics, predictions = backtesting_forecaster_multiseries(
    forecaster        = forecaster,
    series            = data_indexed[[TARGET_COL]],
    cv                = cv,
    levels            = forecaster.levels,
    metric            = "mean_absolute_error",
    verbose           = False,
    suppress_warnings = True
)

In [ ]:
predictions

In [ ]:
# Plotting predictions vs real values in the test set
# ==============================================================================
fig = go.Figure()
trace1 = go.Scatter(x=data_test.index, y=data_test[TARGET_COL], name="test", mode="lines")
trace2 = go.Scatter(
    x=predictions.index,
    y=predictions.loc[predictions["level"] == TARGET_COL, "pred"],
    name="predictions", mode="lines"
)
fig.add_trace(trace1)
fig.add_trace(trace2)
fig.update_layout(
    title="Prediction vs real values in the test set",
    xaxis_title="Date time",
    yaxis_title="O3",
    width=800,
    height=400,
    margin=dict(l=20, r=20, t=35, b=20),
    legend=dict(orientation="h", yanchor="top", y=1.05, xanchor="left", x=0)
)
fig.show()

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error

# 1. Define your specific features
features = [
    'period_enum', 'weekday_enum',
    f'{LAG_SUFFIX}_lag1d', f'{LAG_SUFFIX}_lag2d',
]
target = TARGET_COL

# 2. Scaling (CRITICAL for LSTMs)
scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_x.fit_transform(data_clean[features])
y_scaled = scaler_y.fit_transform(data_clean[[target]])

# 3. Time-Series Split
test_size = 96 * 4
X_train_raw, X_test_raw = X_scaled[:-test_size], X_scaled[-test_size:]
y_train, y_test = y_scaled[:-test_size], y_scaled[-test_size:]

# 4. Reshape to 3D: [Samples, Time Steps, Features]
# Here we treat each row as 1 time step with 12 features
X_train = X_train_raw.reshape((X_train_raw.shape[0], 1, X_train_raw.shape[1]))
X_test = X_test_raw.reshape((X_test_raw.shape[0], 1, X_test_raw.shape[1]))

# 5. Build the LSTM Model
model = Sequential([
    # Input shape is (Time Steps, Features)
    LSTM(64, activation='relu', input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=True),
    Dropout(0.2),
    LSTM(32, activation='relu'),
    Dense(1) # Predicting the single purchase_bid value
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# 6. Train
history = model.fit(
    X_train, y_train, 
    epochs=50, 
    batch_size=32, 
    validation_data=(X_test, y_test),
    verbose=1
)

# 7. Predict and Inverse Scale
y_pred_scaled = model.predict(X_test)
y_pred = scaler_y.inverse_transform(y_pred_scaled)
y_test_actual = scaler_y.inverse_transform(y_test)

# 8. Evaluate
mae = mean_absolute_error(y_test_actual, y_pred)
print(f"LSTM MAE: {mae:.2f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 6))

# Plotting the actual values
plt.plot(y_test_actual, label='Actual Purchase Bid', color='royalblue', linewidth=2, alpha=0.7)

# Plotting the LSTM predictions
plt.plot(y_pred, label='LSTM Forecast', color='darkorange', linewidth=2, linestyle='--')

plt.title(f'LSTM Forecast vs Actuals (Last {test_size//96} Days)', fontsize=14)
plt.xlabel('Time Steps (15-min intervals)', fontsize=12)
plt.ylabel('Purchase Bid Value', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

# Optional: Zoom in on the last 2 days for better clarity
# plt.xlim(192, 384) 

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Calculate baseline MAE (already calculated as 'mae' in your LSTM cell)
baseline_mae = mae
feat_importance = []

print("Calculating permutation importance (this may take a moment)...")

for i, feat in enumerate(features):
    # Create a copy of the test set
    X_test_permuted = X_test.copy()
    
    # Shuffle only the values of the current feature across the test set
    # Since shape is (samples, 1, features), we shuffle index [:, 0, i]
    np.random.shuffle(X_test_permuted[:, 0, i])
    
    # Predict with the 'broken' feature
    y_pred_permuted_scaled = model.predict(X_test_permuted, verbose=0)
    y_pred_permuted = scaler_y.inverse_transform(y_pred_permuted_scaled)
    
    # Calculate new MAE
    permuted_mae = mean_absolute_error(y_test_actual, y_pred_permuted)
    
    # Importance is the increase in error
    importance = permuted_mae - baseline_mae
    feat_importance.append({'Feature': feat, 'Importance': importance})

# 2. Convert to DataFrame and sort
importance_df = pd.DataFrame(feat_importance).sort_values(by='Importance', ascending=False)

# 3. Plot
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importance_df, palette='magma')
plt.title('LSTM Feature Importance (Permutation Method)')
plt.xlabel('Increase in MAE when feature is shuffled')
plt.show()

print(importance_df)

In [ ]:
test_days = 7

# Create a date column (without time) to group by day
data_clean['date'] = data_clean['period_start'].dt.date

# Pivot: Rows = Dates, Columns = The 96 periods
df_daily = data_clean.pivot(index='date', columns='period_enum', values=target)
df_daily = df_daily.dropna() # Ensure we only have full 96-period days

# Scale the entire 96-column matrix
scaler = MinMaxScaler()
df_daily_scaled = scaler.fit_transform(df_daily)

train_daily_scaled = df_daily_scaled[:-test_days]
test_daily_actuals = df_daily_scaled[-test_days:]

# Create X (Day N) and y (Day N+1)
X_train_days = train_daily_scaled[:-1] 
y_train_days = train_daily_scaled[1:]

# Reshape for LSTM: [Samples, Time Steps, Features]
# Features = 96 (the whole day)
X_train_days = X_train_days.reshape((X_train_days.shape[0], 1, 96))
y_train_days = y_train_days.reshape((y_train_days.shape[0], 96))

In [ ]:
print(f"Training on {X_train_days.shape[0]} days.")
print(f"Holding out {test_days} days for testing.")

In [ ]:
X_days.shape, y_days.shape

In [ ]:
from tensorflow.keras.layers import RepeatVector, TimeDistributed

model_vector = Sequential([
    # Input is the 96 values of yesterday
    LSTM(128, activation='relu', input_shape=(1, 96), return_sequences=True),
    LSTM(64, activation='relu'),
    # Output is the 96 values of tomorrow
    Dense(96) 
])

model_vector.compile(optimizer='adam', loss='mse')
model_vector.fit(X_train_days, y_train_days, epochs=100, batch_size=16, verbose=1)

In [ ]:
# Start recursion using the day IMMEDIATELY PRECEDING your test set
current_input = train_daily_scaled[-1:].reshape((1, 1, 96))

test_predictions = []

for day in range(test_days):
    # Predict next day
    next_day_pred = model_vector.predict(current_input, verbose=0)
    test_predictions.append(next_day_pred.flatten())
    
    # Feed prediction back in for the next day's forecast
    current_input = next_day_pred.reshape((1, 1, 96))

# Convert list to array for inverse scaling
test_pred_matrix = np.array(test_predictions)
final_7day_test_forecast = scaler.inverse_transform(test_pred_matrix).flatten()

# Get actual values for the same 7 days for comparison
actual_7day_values = scaler.inverse_transform(test_daily_actuals).flatten()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Get the timestamps for the test period (last 7 days)
test_size = 96 * 7
test_timestamps = data_clean['period_start'].iloc[-test_size:]

# 2. Calculate Evaluation Metrics
mae_7d = mean_absolute_error(actual_7day_values, final_7day_test_forecast)
rmse_7d = np.sqrt(mean_squared_error(actual_7day_values, final_7day_test_forecast))

# 3. Create the Visualization
plt.figure(figsize=(16, 7))

# Plot the Actual Ground Truth
plt.plot(test_timestamps, actual_7day_values, label='Actual Ground Truth', color='black', alpha=0.6, linewidth=1.5)

# Plot the Model's 7-Day Recursive Forecast
plt.plot(test_timestamps, final_7day_test_forecast, label='LSTM Vector Forecast (Recursive)', 
         color='crimson', linestyle='--', linewidth=2)

# Styling and Labels
plt.title(f'7-Day Out-of-Sample Evaluation\nMAE: {mae_7d:.2f} | RMSE: {rmse_7d:.2f}', fontsize=14)
plt.xlabel('Timestamp (15-min intervals)', fontsize=12)
plt.ylabel('Purchase Bid Volume', fontsize=12)
plt.legend(loc='upper right')
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# 4. Print individual day MAEs to see error growth
for d in range(7):
    day_mae = mean_absolute_error(actual_7day_values[d*96:(d+1)*96], 
                                 final_7day_test_forecast[d*96:(d+1)*96])
    print(f"Day {d+1} MAE: {day_mae:.2f}")

In [ ]:
# Start with the very last day in your training set (March 31st)
last_day_vector = df_daily_scaled[-1:].reshape((1, 1, 96))

all_forecasts = []
current_input = last_day_vector

for day in range(7):
    # Predict the entire next day (96 blocks) in one go
    next_day_scaled = model_vector.predict(current_input, verbose=0)
    
    # Store result
    all_forecasts.append(next_day_scaled.flatten())
    
    # Update input: Today's prediction becomes tomorrow's input
    current_input = next_day_scaled.reshape((1, 1, 96))

# Flatten and Inverse Scale
forecast_flat = np.array(all_forecasts).flatten().reshape(-1, 1)
# Create a dummy matrix to inverse scale (since scaler expects 96 columns)
dummy_matrix = np.zeros((len(all_forecasts), 96))
for i in range(len(all_forecasts)):
    dummy_matrix[i] = all_forecasts[i]

final_7day_forecast = scaler.inverse_transform(dummy_matrix).flatten()

In [ ]:
final_7day_forecast

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. Define the timeframe for history (last 7 days)
history_size = 96 * 7 

# 2. Extract historical data for the plot
history_y = data_clean[target].tail(history_size)
history_x = data_clean['period_start'].tail(history_size)

# 3. Create the forecast timeline starting from the last date in data_clean
last_timestamp = data_clean['period_start'].iloc[-1]
forecast_x = pd.date_range(start=last_timestamp + pd.Timedelta(minutes=15), 
                           periods=len(final_7day_forecast), freq='15min')

# 4. Plotting
plt.figure(figsize=(16, 7))

# Plot historical actuals
plt.plot(history_x, history_y, label='Historical Actuals (Last 7 Days)', color='black', alpha=0.6)

# Plot the 7-day Vector LSTM Forecast
plt.plot(forecast_x, final_7day_forecast, label='LSTM Vector Forecast (7-Day Ahead)', 
         color='darkorange', linewidth=2, linestyle='-')

# Add a vertical line to mark the start of the forecast
plt.axvline(x=last_timestamp, color='red', linestyle='--', label='Forecast Start (April 1st)')

plt.title('7-Day Energy Bid Forecast: Historical Context vs. LSTM Vector Prediction', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Purchase Bid Volume', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 1. Pivot all features into daily vectors (lags were precomputed above)
# We create a dictionary of "daily matrices"
feature_cols = [
    target,
    f'{LAG_SUFFIX}_lag1d',
    f'{LAG_SUFFIX}_lag2d',
    f'{LAG_SUFFIX}_lag7d',
    'period_enum',
    'weekday_enum',
]
daily_matrices = {}

for col in feature_cols:
    daily_matrices[col] = data_clean.pivot(index='date', columns='period_enum', values=col).dropna()

# Ensure all matrices have the same dates (intersection)
common_dates = daily_matrices[target].index
for col in feature_cols:
    daily_matrices[col] = daily_matrices[col].loc[common_dates]

# 3. Scale each feature channel individually
scalers = {col: MinMaxScaler() for col in feature_cols}
scaled_matrices = {col: scalers[col].fit_transform(daily_matrices[col]) for col in feature_cols}

In [ ]:
# Features for the model: Lag1, Lag2, Lag7, Period_Enum, Weekday_Enum
# (Note: 'purchase_bid' is our target for the NEXT day)

def create_3d_dataset(matrices, test_days=7):
    X, y = [], []
    # dates = list(matrices[target].index)

    num_days = matrices[target].shape[0]
    # We use matrices['pb_lag1d'], etc., as inputs for day D to predict matrices[target] for day D
    # Actually, to predict Tomorrow (D+1), we use Today's (D) features.
    
    for i in range(num_days - 1):
        # Create a 96x5 slice for the day
        day_features = np.stack([
            scaled_matrices[f'{LAG_SUFFIX}_lag1d'][i],
            scaled_matrices[f'{LAG_SUFFIX}_lag2d'][i],
            scaled_matrices[f'{LAG_SUFFIX}_lag7d'][i],
            scaled_matrices['period_enum'][i],
            scaled_matrices['weekday_enum'][i]
        ], axis=-1) # Shape: (96, 5)
        
        X.append(day_features)
        y.append(scaled_matrices[target][i+1]) # Target: Tomorrow's actual bids
        
    return np.array(X), np.array(y)

X_all, y_all = create_3d_dataset(scaled_matrices)

# Split for testing
X_train, X_test = X_all[:-7], X_all[-7:]
y_train, y_test = y_all[:-7], y_all[-7:]

In [ ]:
from tensorflow.keras.layers import Flatten

model_multi = Sequential([
    # Input shape: (96 time steps, 5 features per step)
    LSTM(128, activation='relu', input_shape=(96, 5), return_sequences=True),
    LSTM(64, activation='relu'),
    Dense(128, activation='relu'),
    Dense(96) # Output: All 96 periods for the next day
])

model_multi.compile(optimizer='adam', loss='mse')
model_multi.fit(X_train, y_train, epochs=100, batch_size=16, verbose=1)

In [ ]:
# Start with the last available training day features
# For a true recursive forecast, you'd need a rolling buffer of the last 7 days of predictions
current_X = X_test[0:1] # Start with the first day of the test period
all_forecasts = []

for d in range(7):
    # 1. Predict tomorrow
    next_day_pred = model_multi.predict(current_X, verbose=0)
    all_forecasts.append(next_day_pred.flatten())
    
    if d < 6:
        # 2. Update the input for the next step
        # This is complex because Lag2 tomorrow = Lag1 today, etc.
        # For simplicity in this vector approach, we update the Lag1 channel 
        # with our prediction and shift the others if you have a buffer.
        
        new_day = current_X.copy()
        # Update Lag1 channel (index 0) with the new prediction
        new_day[0, :, 0] = next_day_pred.flatten() 
        # Update Weekday channel (index 4)
        new_day[0, :, 4] = (new_day[0, :, 4] + (1/7)) % 1.0 
        
        current_X = new_day

# Inverse scale the results
final_forecast = scalers[target].inverse_transform(np.array(all_forecasts)).flatten()

In [ ]:
final_forecast

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 1. Configuration
days_of_history_to_show = 17  # How many days of training data to show before the forecast
test_days = 7
steps_per_day = 96

# 2. Extract Timestamps
# The test period is the last (test_days * 96) rows
test_timestamps = data_clean['period_start'].iloc[-(test_days * steps_per_day):]

# The history period is the segment just before the test period
history_end_idx = len(data_clean) - (test_days * steps_per_day)
history_start_idx = history_end_idx - (days_of_history_to_show * steps_per_day)
history_timestamps = data_clean['period_start'].iloc[history_start_idx:history_end_idx]
history_values = data_clean[target].iloc[history_start_idx:history_end_idx]

# 3. Plotting
plt.figure(figsize=(18, 8))

# Plot Training History (Context)
plt.plot(history_timestamps, history_values, 
         label='Training Data (History)', color='gray', alpha=0.5, linewidth=1)

# Plot Actual Test Values
plt.plot(test_timestamps, actual_7day_values, 
         label='Actual Bids (Test Set)', color='black', linewidth=1.5)

# Plot Multi-Lag Forecast
# Ensure final_forecast is flattened if it isn't already
forecast_plot_values = final_forecast.flatten() if hasattr(final_forecast, 'flatten') else final_forecast
plt.plot(test_timestamps, forecast_plot_values, 
         label='Multi-Lag LSTM Forecast', color='blue', linestyle='--', linewidth=2)

# Add a vertical line to separate Training from Testing
plt.axvline(x=history_timestamps.iloc[-1], color='red', linestyle='-', alpha=0.8, label='Forecast Start')

# Formatting
plt.title(f'7-Day Multi-Lag Vector Forecast\nLags used: [1d, 2d, 7d] + Period Enum', fontsize=16)
plt.xlabel('Date & Time', fontsize=12)
plt.ylabel('Purchase Bid Volume', fontsize=12)
plt.legend(loc='upper left', frameon=True, shadow=True)
plt.grid(True, which='both', linestyle='--', alpha=0.4)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# 4. Final Accuracy Check
from sklearn.metrics import mean_absolute_error
overall_mae = mean_absolute_error(actual_7day_values, forecast_plot_values)
print(f"Overall 7-Day Forecast MAE: {overall_mae:.2f}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error

# 1. Define feature names (order must match your X_train/X_test stacking)
feature_names = [
    f'{LAG_SUFFIX}_lag1d',
    f'{LAG_SUFFIX}_lag2d',
    f'{LAG_SUFFIX}_lag7d',
    'period_enum',
    'weekday_enum',
]

# 2. Calculate Baseline MAE (the error of your current model)
# Using your existing test set and predictions
y_pred_baseline = model_multi.predict(X_test, verbose=0)
baseline_mae = mean_absolute_error(y_test, y_pred_baseline)

importances = []

print("Analyzing feature contributions...")

for i, name in enumerate(feature_names):
    # Create a copy of the test set to mess with
    X_test_permuted = X_test.copy()
    
    # Shuffle the values for this specific feature channel across all days and periods
    # X_test shape is (7, 96, 5) -> we shuffle index [:, :, i]
    # We flatten, shuffle, and reshape to break the temporal relationship
    values = X_test_permuted[:, :, i].flatten()
    np.random.shuffle(values)
    X_test_permuted[:, :, i] = values.reshape(X_test_permuted.shape[0], X_test_permuted.shape[1])
    
    # Predict with the "broken" feature
    y_pred_permuted = model_multi.predict(X_test_permuted, verbose=0)
    permuted_mae = mean_absolute_error(y_test, y_pred_permuted)
    
    # Importance = how much worse the model got
    importance = permuted_mae - baseline_mae
    importances.append(importance)

# 3. Create DataFrame and Plot
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importance_df, palette='viridis')
plt.title('Multi-Lag LSTM Feature Importance\n(Increase in MAE when feature is shuffled)', fontsize=14)
plt.xlabel('Importance (MAE Delta)')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

print(importance_df)

### Persist LSTM training settings for submission forecasts
Stores the Skforecast RNN configuration and last backtest MAE for the active `TARGET_COL`.

In [ ]:
_end_val = data_indexed.index[-HORIZON - 1]
_fit_kw = {
    "epochs": 30,
    "batch_size": 32,
    "callbacks": [EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)],
}
_model = build_default_lstm(series=data_indexed[[TARGET_COL]], lags=96, steps=HORIZON)
_forecaster = ForecasterRnn(
    estimator=_model,
    levels=TARGET_COL,
    lags=96,
    transformer_series=MinMaxScaler(),
    fit_kwargs=_fit_kw,
)
_cv = TimeSeriesFold(
    steps=HORIZON,
    initial_train_size=len(data_indexed.loc[:_end_val, :]),
    refit=False,
)
_metrics, _ = backtesting_forecaster_multiseries(
    forecaster=_forecaster,
    series=data_indexed[[TARGET_COL]],
    cv=_cv,
    levels=[TARGET_COL],
    metric="mean_absolute_error",
    verbose=False,
    suppress_warnings=True,
)
_payload = {
    "target": TARGET_COL,
    "model": "lstm_rnn",
    "horizon": HORIZON,
    "lags": 96,
    "fit_kwargs": {"epochs": 30, "batch_size": 32, "patience": 5},
    "backtest_mae": float(_metrics.iloc[0, 0]),
}
_existing_l = load_json(PARAMS_PATH) if PARAMS_PATH.exists() else {}
_existing_l[TARGET_COL] = _payload
save_json(_existing_l, PARAMS_PATH)
_payload